# LangChain ChatOpenAI + PydanticOutputParser 예제
이 노트북은 LangChain의 `ChatOpenAI`와 `PydanticOutputParser`를 사용하여 구조화된 출력을 생성하는 방법을 보여줍니다.

In [ ]:
# 필수 라이브러리 설치
#uv add pydantic

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
#load_dotenv(dotenv_path='../.env')
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:7])

gsk_DIC


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from pydantic import BaseModel, Field
from typing import List
from pprint import pprint

In [3]:
# 출력 구조를 정의하는 Pydantic 모델
class MovieRecommendation(BaseModel):
    movie_title: str = Field(description="추천 영화 제목")
    reason: str = Field(description="추천 이유")
    genre: List[str] = Field(description="영화 장르")
    estimated_rating: float = Field(description="10점 만점에서 예상 평점")

In [4]:
# Pydantic 출력 파서 초기화
parser = PydanticOutputParser(pydantic_object=MovieRecommendation)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"movie_title": {"description": "추천 영화 제목", "title": "Movie Title", "type": "string"}, "reason": {"description": "추천 이유", "title": "Reason", "type": "string"}, "genre": {"description": "영화 장르", "items": {"type": "string"}, "title": "Genre", "type": "array"}, "estimated_rating": {"description": "10점 만점에서 예상 평점", "title": "Estimated Rating", "type": "number"}}, "required": ["movie_title", "reason", "genre", "estimated_rating"]}
```


In [5]:
# 프롬프트 템플릿 설정
template = """
다음 사용자 요청에 따라 영화를 추천해주세요.
요청: {query}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template)

# 파서의 지시사항을 프롬프트에 주입
prompt = prompt.partial(
    format_instructions=parser.get_format_instructions()
)
pprint(prompt)

ChatPromptTemplate(input_variables=['query'], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"movie_title": {"description": "추천 영화 제목", "title": "Movie Title", "type": "string"}, "reason": {"description": "추천 이유", "title": "Reason", "type": "string"}, "genre": {"description": "영화 장르", "items": {"type": "string"}, "title": "Genre", "type": "array"}, "estimated_rating": {"description": "10점 만점에서 예상 평점", "title": "Estimated Rating", "type": "number"}}, "required": ["movie_title", "reason", "genre", "estimated_ra

In [6]:
# 환경변수에서 OpenAI API 키 로드 (실제 사용시 주석 해제)
# import os
# os.environ["OPENAI_API_KEY"] = "your-api-key"

# ChatOpenAI 모델 초기화
#model = ChatOpenAI(temperature=0.7, model="gpt-3.5-turbo")
model = ChatOpenAI(
    #api_key=OPENAI_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    #model="meta-llama/llama-4-scout-17b-16e-instruct",
    model="moonshotai/kimi-k2-instruct-0905",
    temperature=0.5
)

In [8]:
# 체인 구성 및 실행
query = "2000년대 클래식한 느낌의 공포 영화 추천해줘"
chain = prompt | model | parser
output = chain.invoke({"query": query})

print(type(output))
pprint(output)

<class '__main__.MovieRecommendation'>
MovieRecommendation(movie_title='The Others (2001)', reason='고전적인 고딕 공포 분위기를 잘 살려, 긴장감과 반전이 돋보이는 2000년대 클래식 공포물입니다. 어둡고 차가운 저택을 배경으로 한 심리적 공포가 특징입니다.', genre=['공포', '미스터리', '스릴러'], estimated_rating=8.5)


In [9]:
# 결과 출력
print(f"추천 영화: {output.movie_title}")
print(f"추천 이유: {output.reason}")
print(f"장르: {', '.join(output.genre)}") # List => str 로 변환
print(f"예상 평점: {output.estimated_rating}/10")

추천 영화: The Others (2001)
추천 이유: 고전적인 고딕 공포 분위기를 잘 살려, 긴장감과 반전이 돋보이는 2000년대 클래식 공포물입니다. 어둡고 차가운 저택을 배경으로 한 심리적 공포가 특징입니다.
장르: 공포, 미스터리, 스릴러
예상 평점: 8.5/10


#### PydanticOutputParser를 사용한 Review 분석 예제


In [ ]:
import pprint
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
import os

# Pydantic 모델 정의 (리뷰 분석 결과 구조화)
class ReviewAnalysis(BaseModel):
    """제품 리뷰 분석 결과"""
    rating: float = Field(..., ge=0, le=5, description="5점 만점 예상 평점 (0.0 ~ 5.0)")
    pros: list[str] = Field(..., description="장점 3가지 (리스트)")
    cons: list[str] = Field(..., description="단점 3가지 (리스트)")
    summary: str = Field(..., description="리뷰 한 문장 요약")

# PydanticOutputParser 설정
parser = PydanticOutputParser(pydantic_object=ReviewAnalysis)
format_instructions = parser.get_format_instructions()

print(" 출력 형식 지시사항:")
print(format_instructions)
print("-" * 50)

# ChatPromptTemplate (부분 변수 미리 설정)
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 제품 리뷰 분석 전문가입니다. 
리뷰를 객관적으로 분석하여 다음 형식 **정확히** 따르세요:
{format_instructions}
    
- rating: 5점 만점 실수 (예: 4.2)
- pros/cons: 정확히 3개씩 리스트
- summary: 한 문장"""),
    ("user", "리뷰 내용: {review}")
]).partial(format_instructions=format_instructions)

# Groq LLM 초기화 (재현성 위해 seed 추가)
model = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),  # 환경변수 사용
    base_url="https://api.groq.com/openai/v1",
    model="moonshotai/kimi-k2-instruct-0905",
    temperature=0.1,  # 더 낮춰 안정성 향상
    seed=42  # 재현성 보장!
)

# 체인 생성 (LCEL)
chain = prompt | model | parser

# 테스트 리뷰
review = """
이 스마트폰은 배터리 수명이 정말 좋아서 하루 종일 사용해도 충전이 필요 없었어요. 
카메라 화질도 선명하고, 특히 야간 모드가 훌륭합니다. 
다만 가격이 조금 비싸고, 무게가 200g이 넘어서 손이 피곤할 수 있어요.
"""

print(" 리뷰 분석 중...")
print(f" 리뷰: {review[:100]}...")

# 실행
try:
    output = chain.invoke({"review": review})
    
    print("\n===== 분석 결과 =====")
    print(type(output))
    pprint.pprint(output.dict(), width=80, indent=2)  # Pydantic 객체를 dict로 출력
    
    print(f"\n💬 요약: {output.summary}")
    
except Exception as e:
    print(f" 오류: {e}")

 출력 형식 지시사항:
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "제품 리뷰 분석 결과", "properties": {"rating": {"description": "5점 만점 예상 평점 (0.0 ~ 5.0)", "maximum": 5, "minimum": 0, "title": "Rating", "type": "number"}, "pros": {"description": "장점 3가지 (리스트)", "items": {"type": "string"}, "title": "Pros", "type": "array"}, "cons": {"description": "단점 3가지 (리스트)", "items": {"type": "string"}, "title": "Cons", "type": "array"}, "summary": {"description": "리뷰 한 문장 요약", "title": "Summary", "type": "string"}}, "required": ["rating", "pros", "cons", "summary"]}
```
---------------------------

C:\Users\vega2\AppData\Local\Temp\ipykernel_43792\3267907633.py:64: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  pprint.pprint(output.dict(), width=80, indent=2)  # Pydantic 객체를 dict로 출력


#### PydanticOutputParser를 사용한 날짜시간(DateTime) 예제

In [16]:

import pprint
from datetime import datetime, timedelta
from pydantic import BaseModel, Field, field_validator
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
import os
from zoneinfo import ZoneInfo

# Pydantic 모델
class ExtractedDateTime(BaseModel):
    datetime_str: str = Field(..., description="ISO 8601")
    timezone: str = Field(default="Asia/Seoul")

    @field_validator('datetime_str')
    @classmethod
    def validate_iso(cls, v):
        try:
            datetime.fromisoformat(v.replace('Z', '+00:00'))
            return v
        except:
            raise ValueError("ISO 8601 오류")

parser = PydanticOutputParser(pydantic_object=ExtractedDateTime)

# offset-aware NOW (해결!)
NOW = datetime.now(ZoneInfo("Asia/Seoul"))  # timezone 명시!
print(f" 기준일: {NOW.strftime('%Y-%m-%d %A %H:%M:%S %Z')}")
print(f" 오늘: {NOW.strftime('%Y-%m-%d')}")
print("-" * 60)

def extract_datetime_safely(text: str) -> datetime:
    today_str = NOW.strftime("%Y-%m-%d")
    today_weekday = NOW.strftime("%A")
    
    system_prompt = f"""**현재 날짜: {today_str} ({today_weekday}) 정확 사용!**

규칙:
1. "한달 후" = {today_str} + 30일 → 2026-02-19
2. "다음 주 목요일" = 2026-01-29 (목)
3. "3일 후" = 2026-01-23
4. 시간 없으면 09:00, Asia/Seoul(+09:00)

JSON 예시:
{{"datetime_str": "2026-02-19T14:00:00+09:00", "timezone": "Asia/Seoul"}}"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"텍스트: '{text}'")
    ]
    
    response = model.invoke(messages)
    parsed = parser.parse(response.content)
    
    # offset-aware 변환
    dt = datetime.fromisoformat(
        parsed.datetime_str.replace('Z', '+00:00')
    ).astimezone(ZoneInfo(parsed.timezone))
    
    return dt

# LLM
model = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model="moonshotai/kimi-k2-instruct-0905",
    temperature=0.0,
    seed=42
)

# 테스트
texts = [
    "회의는 한달 후 오후 2시에 예정되어 있습니다.",
    "다음 주 목요일 마감입니다.",
    "3일 후 오후 3시30분 점검.",
    "지난주 금요일 정오에 전시회가 있었습니다."
]

print("\n 2026-01-20 기준 정확 테스트")
print("=" * 70)

for i, text in enumerate(texts, 1):
    print(f"\n[{i}] '{text}'")
    
    dt = extract_datetime_safely(text)
    
    # 둘 다 offset-aware라서 빼기 가능!
    delta = dt - NOW
    print(f"  결과: {dt.strftime('%Y-%m-%d (%A) %H:%M %Z')}")
    print(f"  차이: {delta.days}일 {delta.seconds//3600}시간")


 기준일: 2026-01-20 Tuesday 17:15:50 KST
 오늘: 2026-01-20
------------------------------------------------------------

 2026-01-20 기준 정확 테스트

[1] '회의는 한달 후 오후 2시에 예정되어 있습니다.'
  결과: 2026-02-19 (Thursday) 14:00 KST
  차이: 29일 20시간

[2] '다음 주 목요일 마감입니다.'
  결과: 2026-01-29 (Thursday) 09:00 KST
  차이: 8일 15시간

[3] '3일 후 오후 3시30분 점검.'
  결과: 2026-01-23 (Friday) 15:30 KST
  차이: 2일 22시간

[4] '지난주 금요일 정오에 전시회가 있었습니다.'
  결과: 2026-01-16 (Friday) 12:00 KST
  차이: -5일 18시간
